# 04 — Model Evaluation & Comparison

Brain Tumor MRI Classification · Internship Project

## Setup

In [ ]:
import os, sys, json, shutil
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.config import (CUSTOM_CNN_PATH, MOBILENET_PATH, EFFICIENTNET_PATH,
                        BEST_MODEL_PATH, CLASS_NAMES_JSON,
                        CLASS_NAMES, DISPLAY_NAMES, MODELS_DIR)
from src.preprocessing import build_generators
from src.evaluate      import evaluate_model, compare_models
from src.data_utils    import save_class_names

## Load All Models

In [ ]:
print("Loading models …")
cnn_model = tf.keras.models.load_model(CUSTOM_CNN_PATH)
mv2_model = tf.keras.models.load_model(MOBILENET_PATH)
eff_model = tf.keras.models.load_model(EFFICIENTNET_PATH)
print("All models loaded ✓")

## Evaluate on Test Set

In [ ]:
_, _, test_cnn = build_generators('custom')
_, _, test_mv2 = build_generators('mobilenetv2')
_, _, test_eff = build_generators('efficientnetb0')

r_cnn = evaluate_model(cnn_model, test_cnn, 'Custom CNN')
r_mv2 = evaluate_model(mv2_model, test_mv2, 'MobileNetV2')
r_eff = evaluate_model(eff_model, test_eff, 'EfficientNetB0')

## Model Comparison Table

In [ ]:
df = compare_models([r_cnn, r_mv2, r_eff])
df

## Select & Save Best Model

In [ ]:
best_row   = df.loc[df['F1-score'].idxmax()]
best_name  = best_row['Model']
print(f"\nBest model by F1-score: {best_name}")
print(f"  Accuracy  : {best_row['Accuracy']:.4f}")
print(f"  F1-score  : {best_row['F1-score']:.4f}")

src_path = {
    'Custom CNN':     CUSTOM_CNN_PATH,
    'MobileNetV2':    MOBILENET_PATH,
    'EfficientNetB0': EFFICIENTNET_PATH,
}[best_name]

shutil.copy2(src_path, BEST_MODEL_PATH)
print(f"Best model saved → {BEST_MODEL_PATH}")

# Save class names JSON
save_class_names(CLASS_NAMES)

preproc_map = {
    'Custom CNN':     'custom',
    'MobileNetV2':    'mobilenetv2',
    'EfficientNetB0': 'efficientnetb0',
}
meta = {
    'best_model_name': best_name,
    'preprocessing':   preproc_map[best_name],
    'img_size':        [224, 224],
    'num_classes':     4,
    'class_names':     {str(i): DISPLAY_NAMES[c] for i, c in enumerate(CLASS_NAMES)},
    'test_f1':         float(best_row['F1-score']),
    'test_accuracy':   float(best_row['Accuracy']),
}
import json
meta_path = os.path.join(MODELS_DIR, 'deployment_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"Deployment metadata saved → {meta_path}")

## Final Summary

In [ ]:
print("=" * 60)
print("  FINAL EVALUATION SUMMARY")
print("=" * 60)
for r in [r_cnn, r_mv2, r_eff]:
    print(f"\n{r['model']}:")
    print(f"  Accuracy  : {r['accuracy']:.4f}")
    print(f"  Precision : {r['precision']:.4f}")
    print(f"  Recall    : {r['recall']:.4f}")
    print(f"  F1-score  : {r['f1_score']:.4f}")
print()
print(f"SELECTED DEPLOYMENT MODEL: {best_name}")
print(f"  Path: {BEST_MODEL_PATH}")
print()
print("Run the app:  streamlit run app.py")